# 04 — Backtesting & validation
**Achraf Akiyaf — Master MMSD**

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data_utils import load_masi, audit_and_clean, log_returns, stylized_facts
from src import evt, margin, backtest as bt
from src.garch_evt import fit_garch_evt
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3})

masi = load_masi('../data/raw/masi.csv')
masi, journal = audit_and_clean(masi)
r = log_returns(masi); losses = -r
print('Période:', masi.index.min().date(),'->',masi.index.max().date(),'|',len(masi),'jours')
print('Corrections:', journal)

## 1. Backtesting des marges (Kupiec & Christoffersen)
On compte les fois où la perte réelle 2 jours a dépassé la marge.

In [ ]:
rows = []
for name, model in [('Gaussien','gaussian'),('Historique','hist'),('EVT','evt')]:
    mts = margin.margin_timeseries(r, masi, model, window=500)
    r2 = margin.mpor_returns(r,2).reindex(mts.index)
    viol = (r2 < -mts['margin_pct']).values
    tl = bt.traffic_light(viol, 0.995); tl['Modèle']=name
    tl['marge_moy_MAD']=round(mts.margin_mad.mean())
    rows.append(tl)
df = pd.DataFrame(rows)[['Modèle','color','verdict','obs','expected','kupiec_p','christoffersen_p','marge_moy_MAD']]
df.columns=['Modèle','🚦','Verdict','Viol. obs','Attendu','Kupiec p','Christof. p','Marge moy (MAD)']
df

## 2. Cas d'usage : couverture d'un portefeuille OPCVM
Un fonds de 10 M MAD répliquant le MASI se couvre en vendant des futures.

In [ ]:
V = 10_000_000; idx_now = masi.iloc[-1]
N = V / (idx_now * 10)
print(f'Portefeuille : {V:,} MAD | MASI = {idx_now:,.0f}')
print(f'Contrats à vendre pour couvrir : N = {N:.0f}')
var99,_ = evt.gpd_var_es(evt.fit_gpd(losses, evt.suggest_threshold(losses.values)),0.99)
print(f'VaR99 non couverte : {V*var99/100:,.0f} MAD')
print('Couverture ~parfaite : la position future compense les pertes du portefeuille.')

## Conclusion générale
Le projet démontre, sur 16 ans de données réelles du MASI, que : (1) la loi normale sous-estime le risque extrême ; (2) l'EVT (ξ≈0.3, Fréchet) le capture ; (3) une marge EVT est plus prudente mais procyclique. Prolongements : marge conditionnelle GARCH-EVT, futures sur taux, options.